# 03 · Lab: ingesta a bronze de las tres fuentes

Antes de limpiar o resumir un solo dato, el pipeline tiene que traerlo al lake. Ese primer paso se llama ingesta, y en este lab lo escribes tú. Vas a ingestar las tres fuentes del curso (los viajes de la TLC, las zonas de taxi y el clima de Open-Meteo) a un bronze propio, y lo vas a hacer de una forma que se puede repetir sin miedo: cada ingesta consulta una tabla de control, y si el mes ya se trajo, no lo descarga otra vez, a menos que se lo pidas con `reprocesar`.

Para lograrlo se juntan cuatro piezas, y cada una se explica antes de que la uses. Primero, consumir una fuente: armar la URL del mes y descargar el archivo. Segundo, guardar el archivo crudo intacto, con una copia por cada lote. Tercero, agregar a cada fila unos metadatos que dicen cuándo y de dónde llegó, entre ellos una huella. Cuarto, consultar y registrar en la tabla de control. Esta última pieza ya viene hecha: la ves funcionar paso a paso antes de usarla.

Los ejercicios son cuatro. En el 1 calculas la huella de una fila. En el 2 escribes `ingestar_clima(mes, reprocesar)`, la función completa, con sus cinco pasos. En el 3 escribes `ingestar_zonas(reprocesar)`, que se parece a la anterior pero no tiene mes. En el 4 escribes `ingestar_viajes(mes, reprocesar)`, que descarga un archivo de unos 450 MiB y por eso trae un modo de prueba.

La infraestructura ya está lista: la sesión de Spark, las rutas y las pruebas. Tú escribes solo la lógica, dentro de huecos marcados con `# TU CÓDIGO AQUÍ`. Debajo de cada ejercicio hay una prueba que te dice al instante si tu código está bien, con tres posibles respuestas: `OK` si pasó todo, `PENDIENTE` si todavía queda un hueco sin resolver y `REVISAR`, con una pista, si tu código corre pero algo no coincide con lo esperado.

Cada ejercicio trae además pistas desplegables, ordenadas de la más general a la más concreta. Ábrelas solo cuando las necesites y de una en una. Cada celda de código empieza con un comentario `# Celda N`: es su número dentro de este notebook, y el texto lo usa cuando te manda a una celda concreta.

Lo que escribes aquí no se queda en el notebook: estas tres funciones son las que el DAG de la clase 6 va a llamar, con el mes y el valor de `reprocesar`.

> **Antes de empezar.** En esta máquina solo cabe un proceso de Spark a la vez. Si ya abriste otro notebook de la clase y dejaste Spark encendido, ciérralo primero (cada notebook termina con una celda que apaga Spark) o reinicia el kernel del otro notebook.

## 0. Preparación: el entorno correcto, tu bucket, tus rutas y Spark encendido

Es la misma preparación de los notebooks anteriores, con una celda nueva que define dónde escribe tu ingesta y cómo se conecta a Snowflake. Las celdas ya vienen escritas y las ejecutas sin cambiarlas, salvo el nombre de tu conexión de Snowflake si no es `default`.

### 0.1 Verificar el kernel y Java

El kernel es el programa Python que ejecuta las celdas. Si eliges uno equivocado, las librerías no estarán instaladas. Elige siempre **Python (bigdata)**, que es el entorno `.venv` del proyecto. No elijas los kernels que se llaman *Glue PySpark*, *Glue Spark* o *SparkMagic*: abren una sesión de AWS Glue en la nube y esa sesión cobra mientras está abierta. La celda siguiente imprime qué Python usas y si Java está disponible, porque Spark corre sobre la máquina virtual de Java.

In [ ]:
# Celda 1
# Java: el kernel del notebook no lee ~/.bashrc, así que puede no ver el Java que instaló make setup
# en ~/.local/opt/java17. Si no está en el PATH, lo buscamos ahí y definimos JAVA_HOME antes de usar Spark.
import os, shutil
from pathlib import Path
if shutil.which("java") is None:
    for candidato in [os.environ.get("JAVA_HOME"), str(Path.home() / ".local/opt/java17")]:
        if candidato and (Path(candidato) / "bin/java").exists():
            os.environ["JAVA_HOME"] = candidato
            os.environ["PATH"] = f"{candidato}/bin:" + os.environ["PATH"]
            break

import json, re, subprocess, sys, time, contextlib, gc
from io import StringIO

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este kernel no es el del proyecto (la ruta no contiene .venv). "
          "Elige el kernel «Python (bigdata)» arriba a la derecha y reinicia el notebook.")

try:
    java = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print("Java:", (java.stderr or java.stdout).splitlines()[0])
except FileNotFoundError:
    print("AVISO: no se encontró Java en el PATH; Spark no podrá arrancar.")

import pandas, pyarrow, pyspark
print("pandas", pandas.__version__, "| pyarrow", pyarrow.__version__, "| pyspark", pyspark.__version__)

### 0.2 Tu bucket, armado desde tu cuenta de AWS

Cada estudiante tiene su propio lake en S3. Su nombre no se escribe a mano: se arma con un patrón fijo, `<prefijo>-lake-<número de cuenta>-<región>`, igual que hace la función `lake_bucket` de `scripts/lib.sh` cuando corres un `make`. El número de cuenta se lo preguntamos a AWS con las credenciales del Code Editor (es lo mismo que `aws sts get-caller-identity`). La celda siguiente lo arma, lo imprime y comprueba que existe y que puedes leerlo. En este lab el bucket solo sirve para armar la ruta `s3://<tu bucket>/bronze`, que es la que usará el DAG; hoy tu ingesta escribe en el disco de esta máquina.

In [ ]:
# Celda 2
import boto3
from botocore.exceptions import ClientError

REGION = "us-east-2"

# Raíz del repositorio: la carpeta que contiene pyproject.toml, subiendo desde donde está este notebook.
RAIZ = Path.home() / "bigdata"
for carpeta in [Path.cwd(), *Path.cwd().parents]:
    if (carpeta / "pyproject.toml").exists():
        RAIZ = carpeta
        break
sys.path.insert(0, str(RAIZ / "src"))   # así se puede hacer «from pipeline import ...»

# El prefijo viene del archivo .env del repo (lo escribe «make setup»); si no existe, es «bigdata».
prefijo = os.environ.get("PREFIX")
if not prefijo and (RAIZ / ".env").exists():
    m = re.search(r"^PREFIX=(.+)$", (RAIZ / ".env").read_text(), re.M)
    if m:
        prefijo = m.group(1).strip()
if not prefijo:
    prefijo = "bigdata"

cuenta = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
BUCKET = f"{prefijo}-lake-{cuenta}-{REGION}"
print("Cuenta AWS :", cuenta)
print("Tu bucket  :", BUCKET)

s3 = boto3.client("s3", region_name=REGION)
try:
    s3.head_bucket(Bucket=BUCKET)
    print("Acceso al bucket: correcto")
except ClientError as e:
    print("AVISO: no se pudo leer el bucket:", e.response["Error"]["Code"],
          "- no es grave: la ingesta de este lab escribe en el disco de esta máquina.")

### 0.3 Dónde escribe tu ingesta y cómo se conecta a Snowflake

Las funciones que vas a escribir reciben un parámetro `base_bronze`: la carpeta raíz de tu bronze. Hoy vale `~/datos_clase05/bronze_alumno`, una carpeta de esta máquina, porque aquí Spark solo lee del disco. Cuando el DAG de la clase 6 llame las mismas funciones les pasará `s3://<tu bucket>/bronze` y escribirán en tu lake, sin cambiar una línea de código; la celda siguiente deja las dos rutas listas para que las compares.

La tabla de control vive en Snowflake y las funciones `consultar_watermark` y `registrar_watermark` ya están hechas en `src/pipeline/watermark.py`. Se conectan con la conexión de `~/.snowflake/connections.toml` llamada `default`: es el archivo que guarda tu cuenta, tu usuario y tu llave de Snowflake, y lo dejó listo la configuración del curso. Si la tuya se llama distinto, cambia el nombre en la celda siguiente. Las funciones fijan solas el rol `ACCOUNTADMIN` y el warehouse `COMPUTE_WH`. No crean nada: el esquema `BIGDATA.CONTROL` y la tabla ya existen porque los creó `make snowflake` (Terraform) al comienzo de la clase. La celda siguiente comprueba que la conexión funciona y que la tabla está, antes de que empieces los ejercicios.

Una advertencia que cuenta para la clase 6: dentro de Glue no existe el archivo `connections.toml`, así que cómo llega la conexión a Snowflake desde un job de Glue es un problema que todavía hay que resolver. Por ahora estas funciones se usan desde el Code Editor.

In [ ]:
# Celda 3
import urllib.request, hashlib, tempfile, shutil
from datetime import datetime, timezone

from pipeline import bronze as bronze_pipeline   # URLs de las fuentes y piezas de ingesta que ya tiene el pipeline
from pipeline import io as pio                   # rutas del lake: crudo, lotes y marcador de lote completo
from pipeline import watermark                   # la tabla de control, ya hecha
from pipeline.watermark import consultar_watermark, registrar_watermark, borrar_registros_de_prueba, MES_SIN_MES

DATOS = Path(os.environ.get("CLASE05_DATOS", Path.home() / "datos_clase05"))
MB = 1024 * 1024

# Dónde escribe tu ingesta: una carpeta de esta máquina hoy; la ruta de S3 la usará el DAG.
BRONZE_ALUMNO = str(DATOS / "bronze_alumno")
BRONZE_S3 = f"s3://{BUCKET}/bronze"

# Nombre de tu conexión de Snowflake (~/.snowflake/connections.toml). Cámbialo si el tuyo es distinto.
CONEXION_SNOWFLAKE = "default"
os.environ["WATERMARK_CONNECTION"] = CONEXION_SNOWFLAKE     # así lo lee pipeline.watermark

print("Tu bronze de práctica        :", BRONZE_ALUMNO)
print("Bronze en S3 (lo usa el DAG) :", BRONZE_S3)
print("Conexión de Snowflake        :", CONEXION_SNOWFLAKE)
print("Tabla de control             :", watermark.TABLA_CONTROL)

# Comprobar Snowflake ahora, y no en medio de un ejercicio: una consulta a la tabla de práctica «demo».
try:
    consultar_watermark("demo", "bronze", "2099-12")
    print("\nSnowflake: conexión correcta y tabla de control encontrada.")
except Exception as error:
    print("\nSnowflake NO responde:", type(error).__name__, "-", str(error)[:200])
    print("Arréglalo antes de los ejercicios: en la terminal, cd ~/bigdata && make snowflake,")
    print("y revisa que CONEXION_SNOWFLAKE tenga el nombre de tu conexión.")

### 0.4 Dos utilidades pequeñas

Para no repetir código, la celda siguiente define un cronómetro que imprime cuánto tardó un bloque y una función `fmt` que escribe los números grandes con espacios entre los miles, como en el resto del curso.

In [ ]:
# Celda 4
TIEMPOS = {}


@contextlib.contextmanager
def cronometro(etiqueta):
    """Mide un bloque, lo imprime y lo guarda en TIEMPOS."""
    t0 = time.perf_counter()
    yield
    TIEMPOS[etiqueta] = time.perf_counter() - t0
    print(f"[{etiqueta}] {TIEMPOS[etiqueta]:.2f} s")


def fmt(n):
    """19663930 -> '19 663 930'"""
    return f"{n:,}".replace(",", " ")

### 0.5 Arrancar Spark con la misma configuración del pipeline

No vamos a inventar una configuración aparte para el notebook: usamos la misma función `get_spark("local")` de `src/pipeline/engine.py` que usa el pipeline. Crea una sesión `local[*]`, que significa que todo vive en un solo proceso de Java: el driver y un único ejecutor, que tiene tantos hilos como núcleos tenga la máquina (dos en el Code Editor). El driver tiene 2 GB de memoria, la zona horaria de la sesión queda en UTC y los reemplazos de particiones son dinámicos.

Después de arrancar, la celda imprime `defaultParallelism`, el número de tareas que Spark puede ejecutar al mismo tiempo, y el enlace a la Spark UI, la página donde Spark muestra cada trabajo, etapa y tarea mientras corre.

In [ ]:
# Celda 5
# La Spark UI se ve a través del proxy del Code Editor, que agrega el prefijo /codeeditor/default/ports/4040.
# Spark necesita saberlo para armar sus enlaces, estilos e imágenes con ese prefijo; si no, la página
# carga sin diseño. Se indica antes de arrancar Spark, porque se lee al encender la JVM.
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.proxyBase=/codeeditor/default/ports/4040 pyspark-shell"

from pyspark.sql import functions as F, DataFrame, Window
from pipeline.engine import get_spark

with cronometro("arrancar Spark"):
    spark = get_spark("local")

print("Versión de Spark :", spark.version)
print("Master           :", spark.sparkContext.master)
print("Memoria del driver:", spark.sparkContext.getConf().get("spark.driver.memory"))
print("defaultParallelism (tareas simultáneas):", spark.sparkContext.defaultParallelism)


def url_spark_ui():
    """Enlace a la Spark UI. Dentro del Code Editor pasa por el proxy del Space, igual que Airflow."""
    local = spark.sparkContext.uiWebUrl            # p. ej. http://10.0.0.5:4040
    puerto = local.rsplit(":", 1)[-1].strip("/")
    meta = Path("/opt/ml/metadata/resource-metadata.json")
    if meta.exists():
        try:
            datos = json.loads(meta.read_text())
            r = boto3.client("sagemaker", region_name=REGION).describe_space(
                DomainId=datos["DomainId"], SpaceName=datos["SpaceName"])
            base = re.match(r"https://[^/]+", r["Url"]).group(0)
            return f"{base}/codeeditor/default/ports/{puerto}/jobs/"   # /jobs/ directo: la raíz redirige mal detrás del proxy
        except Exception as e:                      # sin permiso o fuera del Code Editor
            print("(no se pudo consultar el Space:", type(e).__name__, ")")
    return f"{local}   (en el Code Editor: https://<id-del-space>.studio.{REGION}.sagemaker.aws/codeeditor/default/ports/{puerto}/)"


print("Spark UI:", url_spark_ui())

**Qué observar.** `defaultParallelism` es el número de hilos del único ejecutor de esta máquina; esa cifra manda cuántas tareas avanzan a la vez. Abre el enlace de la Spark UI en otra pestaña y déjalo abierto: la pestaña *Jobs* está vacía porque Spark todavía no ha hecho nada, y se irá llenando a medida que ejecutes celdas. La UI solo existe mientras la sesión de Spark está viva.

## 1. Consumir una fuente: armar la URL del mes y descargar el archivo

Consumir una fuente es pedirle a un sistema externo los datos que publica. Las tres fuentes de este curso se piden por HTTP: el pipeline escribe una URL, el servidor responde con un archivo y el pipeline lo guarda. Lo que cambia de una fuente a otra es qué entrega la URL y cómo se pide un mes concreto.

| Fuente | Qué entrega | Cómo se pide un mes |
|---|---|---|
| Viajes (TLC FHVHV) | Un archivo Parquet de unos 450 MiB. | Hay un archivo por mes y el mes va en el nombre: `https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet`. |
| Zonas de taxi | Un CSV de 265 filas. | No tiene mes: es un solo archivo, `https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv`. |
| Clima (Open-Meteo) | Un JSON con la lluvia y la temperatura diarias de NYC. | No hay un archivo por mes: la URL lleva `start_date` y `end_date`, y la API arma el JSON con ese rango. |

Los viajes y las zonas son archivos ya publicados, y basta con descargarlos. El clima es una consulta: para pedir un mes hay que calcular su primer y su último día, y por eso `bronze.url_clima(mes)` usa `ultimo_dia_mes`, que respeta los meses de 28, 29, 30 y 31 días. El mes siempre se escribe como texto `AAAA-MM`.

La celda siguiente arma las tres URL para febrero de 2024, le pregunta al servidor de los viajes cuánto pesa el archivo sin descargarlo (una petición `HEAD` pide solo el encabezado) y descarga de verdad el clima y las zonas, que son pequeños, para que veas qué llega.

In [ ]:
# Celda 6
def url_viajes(mes):
    """Ya resuelta: URL del archivo Parquet de viajes del mes (AAAA-MM)."""
    return bronze_pipeline.URL_VIAJES.format(mes=mes)


def tamano_remoto_mb(url):
    """Ya resuelta: cuánto pesa el archivo, preguntándole al servidor solo por el encabezado (HEAD)."""
    peticion = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(peticion, timeout=60) as respuesta:
        return int(respuesta.headers["Content-Length"]) / MB


def descargar_en_memoria(url):
    """Ya resuelta: descarga una URL pequeña y devuelve sus bytes tal cual llegan."""
    with urllib.request.urlopen(url, timeout=60) as respuesta:
        return respuesta.read()


mes = "2024-02"
print("Viajes :", url_viajes(mes))
print("Zonas  :", bronze_pipeline.URL_ZONAS)
print("Clima  :", bronze_pipeline.url_clima(mes))
print(f"\nEl archivo de viajes de {mes} pesa {tamano_remoto_mb(url_viajes(mes)):,.0f} MiB (no se descargó).")

clima_en_bytes = descargar_en_memoria(bronze_pipeline.url_clima(mes))
print(f"\nEl clima de {mes} son {len(clima_en_bytes):,} bytes. Así empieza el JSON:")
print(clima_en_bytes[:330].decode("utf-8"))

zonas_en_bytes = descargar_en_memoria(bronze_pipeline.URL_ZONAS)
print(f"\nLas zonas son {len(zonas_en_bytes):,} bytes. Así empiezan:")
print(zonas_en_bytes[:200].decode("utf-8"))

**Qué observar.** El clima de febrero termina en `2024-02-29`: 2024 es bisiesto y la función lo calculó sola. Cambia `mes` por `2023-02` y vuelve a ejecutar la celda para ver que esa vez termina en el día 28. El archivo de viajes pesa más de 400 MiB, y esa es la razón por la que la ingesta debe saber si un mes ya se trajo: descargarlo dos veces por descuido cuesta tiempo y le pide dos veces lo mismo al servidor. El JSON del clima trae listas dentro de `daily` (una por variable), y las zonas son texto con encabezado; eso lo resolverá cada función de ingesta al leer el archivo con Spark.

## 2. Guardar el crudo intacto, una copia por cada lote

El primer destino de una descarga es el archivo crudo: los mismos bytes que entregó la fuente, sin leerlos, sin convertirlos y sin corregirlos. Esa copia fiel es la base de todo lo demás. Si dentro de dos años alguien duda de un número, se vuelve a ese archivo; si cambia una regla de limpieza, se reconstruye todo desde él sin pedirle nada a la fuente.

Cada ejecución de la ingesta crea un *lote*, una carpeta nueva con un identificador propio, y nunca reemplaza a los lotes anteriores. El crudo también se guarda por lote. La razón es que una fuente puede corregir un archivo entre dos descargas: si el crudo de febrero se sobrescribiera, el lote viejo perdería el archivo del que salió. Con una copia por lote, cada Parquet de bronze conserva el crudo exacto que lo produjo. Este es el diseño, con el clima y las zonas:

```text
<bronze>/
  _crudo/clima/mes=2024-01/lote=20261117T143005Z_a3f9c1/open_meteo_2024-01.json   crudo del lote, tal como llegó
  clima/mes=2024-01/lote=20261117T143005Z_a3f9c1/                                  Parquet con metadatos + _LOTE_COMPLETO
  clima/mes=2024-01/lote=20261118T091210Z_07be44/                                  segunda ingesta del mes: otro lote
  _crudo/taxi_zones/lote=20261117T143005Z_a3f9c1/taxi_zone_lookup.csv              las zonas no tienen carpeta de mes
  taxi_zones/lote=20261117T143005Z_a3f9c1/
```

El identificador del lote, por ejemplo `20261117T143005Z_a3f9c1`, es la fecha y la hora UTC de la ingesta seguidas de seis caracteres hexadecimales al azar. Como empieza con la fecha, ordenar los identificadores como texto los ordena en el tiempo, y los seis caracteres evitan un choque entre dos ingestas del mismo segundo. Dentro de la carpeta del lote, el Parquet se escribe y, solo cuando terminó completo, se publica un archivo vacío llamado `_LOTE_COMPLETO`. Silver lee siempre el lote más reciente que tenga ese marcador, así que un lote que quedó a medias porque la ingesta falló se ignora. Se usa un marcador propio porque Glue no siempre escribe el `_SUCCESS` de Spark.

La descarga misma también se protege: `bronze.descargar(url, destino)` baja el archivo a un archivo temporal y lo publica en `destino` solo cuando terminó. Funciona igual si `destino` es una carpeta local o una ruta `s3://`. La celda siguiente define `ruta_crudo_lote`, la función que arma la ruta del crudo de un lote, y muestra las rutas de un lote de ejemplo.

In [ ]:
# Celda 7
def ruta_crudo_lote(base, tabla, id_lote, nombre, mes=None):
    """Ya resuelta: ruta del crudo de un lote, <base>/_crudo/<tabla>[/mes=AAAA-MM]/lote=<id_lote>/<nombre>."""
    carpeta_de_la_tabla = pio.dir_crudo(base, tabla, mes)
    return pio.unir(carpeta_de_la_tabla, f"lote={id_lote}", nombre)


ahora = datetime.now(timezone.utc).replace(microsecond=0)
id_de_ejemplo = bronze_pipeline.nuevo_id_lote(ahora)
nombre_clima = pio.nombre_clima("2024-01")
nombre_viajes = pio.nombre_viajes("2024-01")

print("Identificador del lote:", id_de_ejemplo, "\n")
print("Crudo del clima :", ruta_crudo_lote("<bronze>", pio.CLIMA, id_de_ejemplo, nombre_clima, "2024-01"))
print("Lote del clima  :", pio.dir_lote("<bronze>", pio.CLIMA, id_de_ejemplo, "2024-01"))
print("Crudo de zonas  :", ruta_crudo_lote("<bronze>", pio.ZONAS, id_de_ejemplo, pio.NOMBRE_ZONAS))
print("Lote de zonas   :", pio.dir_lote("<bronze>", pio.ZONAS, id_de_ejemplo))
print("Crudo de viajes :", ruta_crudo_lote("<bronze>", pio.VIAJES, id_de_ejemplo, nombre_viajes, "2024-01"))
print("Lote de viajes  :", pio.dir_lote("<bronze>", pio.VIAJES, id_de_ejemplo, "2024-01"))

**Qué observar.** El crudo y el lote comparten el mismo identificador, así que dado un lote se sabe exactamente de qué archivo salió. La carpeta del mes (`mes=2024-01`) existe para el clima y los viajes, y no para las zonas. Las rutas empiezan con `<bronze>` porque todavía no estamos escribiendo nada: cada función de ingesta recibirá la raíz verdadera en el parámetro `base_bronze`.

## 3. Los metadatos de cada fila y la huella

El Parquet de un lote lleva, además de las columnas originales de la fuente, cinco columnas de metadatos que dejan escrito cuándo y de dónde llegó cada fila:

| Columna | Qué guarda | Para qué sirve |
|---|---|---|
| `_fecha_ingesta` | El momento exacto, en UTC, en que arrancó la ingesta del lote. | Saber qué tan fresco es el dato y ordenar lotes en el tiempo. |
| `_id_lote` | El identificador del lote, por ejemplo `20261117T143005Z_a3f9c1`. | Distinguir una ingesta de otra y encontrar su carpeta de crudo. |
| `_hash_fila` | La huella SHA-256 de la fila: un texto de 64 caracteres calculado con todas las columnas originales. | Detectar filas repetidas y filas cambiadas sin comparar columna por columna. Dos filas con los mismos valores dan la misma huella. |
| `_fuente` | La URL de donde se descargó el dato. | Saber de dónde viene cada fila. |
| `_archivo_origen` | El nombre del archivo crudo del que salió la fila. | Volver al crudo y reproducir el resultado. |

Estas columnas viajan dentro del archivo, y no solo en el nombre de la carpeta: si alguien copia el Parquet a otro sitio, sigue sabiendo de qué lote es.

La huella se calcula así. Cada columna original se convierte a texto, el nulo se reemplaza por el texto `<nulo>` (para que un nulo no se confunda con un texto vacío), los textos se unen con `||` y a ese texto se le aplica SHA-256. Los metadatos no entran en la huella, y por eso el mismo dato ingestado dos veces tiene la misma huella aunque tenga otro lote y otra fecha. En la clase 6 esa huella se usará para quitar los duplicados que aparecen cuando se juntan varios lotes del mismo mes.

Antes del primer ejercicio, la celda siguiente deja las herramientas de las pruebas. La función `verificar` ejecuta la prueba de cada ejercicio e imprime una de tres cosas: `OK`, `PENDIENTE` si queda un hueco sin resolver (una línea con `...  # TU CÓDIGO AQUÍ`) o `REVISAR` con una pista. También define `huella_en_python`, que escribe la definición de la huella con `hashlib` para comparar contra ella.

In [ ]:
# Celda 8
import inspect
from pyspark.sql import Column

from pipeline.bronze import MARCADOR_NULO, SEPARADOR_HASH, COLUMNAS_METADATOS

FORMATO_ID_LOTE = re.compile(r"^\d{8}T\d{6}Z_[0-9a-f]{6}$")
RESULTADO_LAB = {}
HUECO = re.compile(r"(=|^)\s*\.\.\.\s*#\s*TU CÓDIGO AQUÍ", re.M)


def verificar(nombre, funcion, prueba):
    """Corre prueba(funcion) e imprime OK, PENDIENTE o REVISAR con una pista."""
    try:
        fuente = inspect.getsource(funcion)
    except (OSError, TypeError):
        fuente = ""
    if HUECO.search(fuente):
        estado, detalle = "PENDIENTE", "todavía hay un hueco «...  # TU CÓDIGO AQUÍ» sin resolver."
    else:
        try:
            prueba(funcion)
            estado, detalle = "OK", ""
        except AssertionError as e:
            estado, detalle = "REVISAR", str(e)
        except Exception as e:
            estado, detalle = "REVISAR", f"tu función falló con {type(e).__name__}: {str(e)[:300]}"
    RESULTADO_LAB[nombre] = estado
    if detalle:
        print(f"{nombre}: {estado}  {detalle}")
    else:
        print(f"{nombre}: {estado}")


def huella_en_python(valores):
    """La definición de la huella escrita con hashlib, para comparar: texto, nulo -> MARCADOR_NULO, unir, SHA-256."""
    partes = []
    for valor in valores:
        if valor is None:
            partes.append(MARCADOR_NULO)
        else:
            partes.append(str(valor))
    return hashlib.sha256(SEPARADOR_HASH.join(partes).encode("utf-8")).hexdigest()

### Ejercicio 1 · Bronze: la huella de una fila

Escribes tu código en la Celda 9 y la Celda 10 lo verifica.

**Qué hacer.** Completa `calcular_hash_fila(columnas)`. Recibe la lista de nombres de las columnas originales y devuelve una **columna de Spark** con la huella SHA-256 de cada fila, escrita en texto hexadecimal. Los pasos son los de la explicación anterior: convertir cada columna a texto con `cast("string")`, reemplazar el nulo por `MARCADOR_NULO` con `F.coalesce` y `F.lit`, unir los textos con `SEPARADOR_HASH` usando `F.concat_ws`, y aplicar `F.sha2(..., 256)`.

**Por qué importa.** Una huella permite saber si dos filas son idénticas comparando un solo texto de 64 caracteres. Se reemplaza el nulo explícitamente porque `concat_ws` ignora los nulos: sin ese reemplazo, las filas `("a", nulo)` y `("a", "")` darían la misma huella, y también `(nulo, "a")` y `("a", nulo)`. El orden de las columnas también cuenta, por eso la función recibe una lista.

<details>
<summary>Pista 1: la idea</summary>

Para cada columna haces dos cosas seguidas: la conviertes a texto y reemplazas el nulo por el marcador. Con todos los textos listos, los unes con el separador y calculas la huella de esa fila. El primer hueco está dentro del `for`; los otros dos, después de él.

</details>

<details>
<summary>Pista 2: qué función va en cada hueco</summary>

`F.coalesce(a, b)` devuelve `a` y, si es nulo, devuelve `b`. `F.lit(x)` convierte el valor `x` en una columna. `F.concat_ws(sep, *columnas)` une una lista de columnas con el separador `sep`; como `textos` es una lista de Python, hay que desempaquetarla con `*`. `F.sha2(columna, 256)` devuelve la huella en hexadecimal.

</details>

In [ ]:
# Celda 9
def calcular_hash_fila(columnas: list[str]) -> Column:
    """SHA-256 (hexadecimal) de la fila: valores como texto, nulo como MARCADOR_NULO, unidos con SEPARADOR_HASH."""
    textos = []
    for nombre in columnas:
        como_texto = F.col(nombre).cast("string")
        # concat_ws se salta los nulos; con el marcador, un nulo ocupa su lugar y no se confunde con un texto vacío.
        sin_nulos = ...  # TU CÓDIGO AQUÍ: si como_texto es nulo, usa el texto MARCADOR_NULO (F.coalesce y F.lit)
        textos.append(sin_nulos)
    fila_unida = ...  # TU CÓDIGO AQUÍ: une los textos con SEPARADOR_HASH (F.concat_ws)
    huella = ...  # TU CÓDIGO AQUÍ: SHA-256 de fila_unida (F.sha2 con 256 bits)
    return huella

In [ ]:
# Celda 10
def huella_en_python(valores):
    """La definición de la huella escrita con hashlib, para comparar: texto, nulo -> MARCADOR_NULO, unir, SHA-256."""
    partes = []
    for valor in valores:
        if valor is None:
            partes.append(MARCADOR_NULO)
        else:
            partes.append(str(valor))
    return hashlib.sha256(SEPARADOR_HASH.join(partes).encode("utf-8")).hexdigest()


def prueba_e1(f):
    # 1. Cinco filas con nulos y vacíos: cada una debe dar la huella de la definición.
    datos = [("a", "b"), ("a", None), ("a", ""), (None, "a"), ("a", None)]
    df = spark.createDataFrame(datos, "x string, y string").withColumn("h", f(["x", "y"]))
    for fila in df.collect():
        esperada = huella_en_python([fila["x"], fila["y"]])
        assert fila["h"] == esperada, (f"Para la fila {(fila['x'], fila['y'])} la huella debería ser {esperada[:12]}... y es {str(fila['h'])[:12]}... "
                                       f"(texto + nulo como «{MARCADOR_NULO}» + unir con «{SEPARADOR_HASH}» + SHA-256).")
    # 2. Una fila con nulo, otra con texto vacío y otra con el nulo en otro lugar no pueden confundirse.
    assert distinct_ok(df), "(«a», nulo), («a», «») y (nulo, «a») deben dar huellas distintas: reemplaza el nulo por el marcador."
    # 3. Los números se convierten a texto igual que en el pipeline.
    numeros = spark.createDataFrame([(1, 2.5, "x")], "n int, d double, t string").withColumn("h", f(["n", "d", "t"])).first()
    assert numeros["h"] == huella_en_python([1, 2.5, "x"]), "Con números la huella debe ser la de «1||2.5||x»: convierte cada columna a texto con cast(\"string\")."
    # 4. El orden de las columnas importa y el resultado es igual al de la función del pipeline.
    ab = spark.createDataFrame([("a", "b")], "x string, y string")
    h_xy = ab.select(f(["x", "y"]).alias("h")).first()["h"]
    h_yx = ab.select(f(["y", "x"]).alias("h")).first()["h"]
    assert h_xy != h_yx, "El orden de las columnas debe cambiar la huella."
    assert h_xy == ab.select(bronze_pipeline.hash_fila(["x", "y"]).alias("h")).first()["h"], "Tu huella debe ser igual a la de bronze.hash_fila del pipeline."


def distinct_ok(df):
    huellas = []
    for fila in df.select("x", "y", "h").distinct().collect():
        huellas.append((fila["x"], fila["y"], fila["h"]))
    distintas = set()
    for x, y, h in huellas:
        distintas.add(h)
    return len(distintas) == len(huellas)


verificar("E1 calcular_hash_fila", calcular_hash_fila, prueba_e1)

Con tu huella lista, la celda siguiente define `agregar_metadatos`, que ya viene resuelta. Recibe un DataFrame con las columnas originales y le agrega las cinco columnas de metadatos, en el orden de `COLUMNAS_METADATOS`. La huella se calcula primero, solo con las columnas originales, y por eso no depende de las otras cuatro. Es la misma lógica de `bronze.agregar_metadatos` del pipeline, pero usando **tu** `calcular_hash_fila`.

In [ ]:
# Celda 11
def agregar_metadatos(df, fuente, archivo_origen, id_lote, fecha_ingesta):
    """Ya resuelta: agrega _hash_fila (tu función), _fecha_ingesta, _id_lote, _fuente y _archivo_origen."""
    originales = df.columns
    con_huella = df.withColumn("_hash_fila", calcular_hash_fila(originales))
    con_metadatos = (
        con_huella
        .withColumn("_fecha_ingesta", F.lit(fecha_ingesta).cast("timestamp"))
        .withColumn("_id_lote", F.lit(id_lote))
        .withColumn("_fuente", F.lit(fuente))
        .withColumn("_archivo_origen", F.lit(archivo_origen))
    )
    # Primero las columnas originales y después los metadatos, siempre en el mismo orden.
    return con_metadatos.select(*originales, *COLUMNAS_METADATOS)

## 4. La tabla de control: qué ya se ingestó, y reprocesar

Una ingesta que corre cada mes tiene que contestar una pregunta antes de descargar nada: ¿este mes ya lo traje? Si la respuesta solo vive en la cabeza de quien ejecuta, falla cuando alguien repite una corrida, cuando el DAG reintenta una tarea o cuando dos personas ingestan lo mismo. Por eso la respuesta se guarda en una tabla de control, también llamada *watermark* (marca de agua): la marca de hasta dónde ya se trajo.

La tabla vive en Snowflake con el nombre `BIGDATA.CONTROL.WATERMARK`, tiene una fila por cada ingesta terminada, y la crea Terraform (`make snowflake`) antes de la clase.

| Columna | Qué guarda | Ejemplo |
|---|---|---|
| `FUENTE` | La URL de donde se descargó. | `https://...fhvhv_tripdata_2024-01.parquet` |
| `CAPA` | La capa del lake que se escribió. | `bronze` |
| `TABLA` | La carpeta de esa capa. | `clima`, `taxi_zones`, `fhvhv` |
| `MES` | El mes en texto `AAAA-MM`. Las zonas no tienen mes y usan el valor fijo `sin_mes`. | `2024-01` |
| `FILAS` | Cuántas filas quedaron escritas en el lote. | `31` |
| `INICIO` | El momento, en UTC, en que arrancó la ingesta. | `2026-11-17 14:30:05` |
| `REPROCESAR` | Verdadero si la ingesta se pidió con reprocesar. | `false` |

Hay dos métodos, que ya están hechos en `src/pipeline/watermark.py`. `consultar_watermark(tabla, capa, mes)` devuelve un texto en formato JSON con la lista de todos los registros de esa tabla, capa y mes, ordenados por `INICIO`; si no hay ninguno, devuelve `[]`. `registrar_watermark(fuente, capa, tabla, mes, filas, inicio, reprocesar)` agrega una fila. Las dos hablan con Snowflake con consultas parametrizadas: los valores viajan aparte de la sentencia SQL, nunca pegados en ella.

La regla de la ingesta es corta. Si la tabla ya tiene un registro del mes y `reprocesar` es falso, la función imprime por qué no hace nada y termina. Si no hay registro, o si `reprocesar` es verdadero, descarga, escribe un lote nuevo y, **solo al terminar bien**, registra. Que el registro vaya al final es lo que hace confiable la tabla: si la ingesta falla a la mitad, no queda registro, y la siguiente corrida lo intenta de nuevo. Reprocesar tampoco borra nada: crea un lote nuevo y una fila nueva, los anteriores siguen ahí, y silver lee el lote más reciente. Se reprocesa, por ejemplo, cuando la fuente corrigió su archivo o cuando una ingesta anterior salió mal.

Antes de pedirte que los uses, mira los dos métodos funcionando. Las tres celdas siguientes usan una tabla llamada `demo`, que no es ninguna fuente real, y la última la limpia. La primera consulta un mes que no existe y la primera vez tarda un poco más, porque crea el esquema y la tabla.

In [ ]:
# Celda 12
# Para poder repetir la demostración, se parte sin registros de la tabla «demo».
borrar_registros_de_prueba("demo")

# Consultar un mes que nunca se ingestó: no hay registros, y la respuesta es una lista vacía en formato JSON.
respuesta = consultar_watermark("demo", "bronze", "2099-12")
print("Respuesta:", repr(respuesta))
print("Tipo     :", type(respuesta).__name__)
print("Registros:", len(json.loads(respuesta)))

**Qué observar.** La respuesta es el texto `[]`, no una lista de Python: es JSON, y para recorrerlo se convierte con `json.loads`. Una lista vacía significa «este mes no se ha ingestado». La celda siguiente registra una ingesta de ejemplo y consulta otra vez.

In [ ]:
# Celda 13
# Registrar una ingesta de ejemplo de la tabla «demo»: 31 filas del mes 2024-01.
inicio_de_ejemplo = datetime.now(timezone.utc)
registrar_watermark(
    fuente="https://ejemplo.org/datos_2024-01.json",   # de dónde se descargó
    capa="bronze",                                     # qué capa se escribió
    tabla="demo",                                      # qué tabla de esa capa
    mes="2024-01",                                     # qué mes
    filas=31,                                          # cuántas filas quedaron escritas
    inicio=inicio_de_ejemplo,                          # cuándo arrancó la ingesta (UTC)
    reprocesar=False,                                  # ¿se pidió reprocesar?
)

respuesta = consultar_watermark("demo", "bronze", "2024-01")
print("Texto JSON tal como llega:")
print(respuesta)

registros = json.loads(respuesta)
print("\nYa convertido a una lista de Python, con un diccionario por registro:")
print(json.dumps(registros, indent=2, ensure_ascii=False))

**Qué observar.** Cada registro es un diccionario con siete campos, uno por columna de la tabla, en minúscula. `fuente` es la URL, `capa` y `tabla` dicen qué se escribió, `mes` dice de qué mes es, `filas` cuenta lo que quedó escrito, `inicio` es el momento de arranque en UTC y `reprocesar` dice si esa ingesta se pidió con reprocesar. La celda siguiente simula un reproceso: registra una segunda ingesta del mismo mes con `reprocesar=True`, para que veas que la consulta devuelve las dos, la más antigua primero.

In [ ]:
# Celda 14
registrar_watermark("https://ejemplo.org/datos_2024-01.json", "bronze", "demo", "2024-01", 31,
                    datetime.now(timezone.utc), True)

registros = json.loads(consultar_watermark("demo", "bronze", "2024-01"))
print(f"Registros de demo / 2024-01: {len(registros)}")
for registro in registros:
    print(f"  inicio={registro['inicio']}  filas={registro['filas']}  reprocesar={registro['reprocesar']}")

# La decisión que tomará cada ingesta con esta respuesta:
hay_registro = len(registros) > 0
for reprocesar in [False, True]:
    if hay_registro and not reprocesar:
        decision = "no hace nada y avisa por qué"
    else:
        decision = "descarga, escribe un lote nuevo y registra"
    print(f"Con reprocesar={reprocesar}: {decision}")

# Limpieza: solo las tablas que empiezan con «demo» o «prueba» se pueden borrar con este método.
borradas = borrar_registros_de_prueba("demo")
print(f"\nSe borraron {borradas} registros de demo. La tabla de las fuentes reales no se tocó.")

**Qué observar.** Con dos registros del mismo mes, el último de la lista es el vigente. La decisión no depende de cuántos registros haya, sino de si hay alguno y de `reprocesar`. Reprocesar agrega una fila y no toca las anteriores: la historia de lo que se ingestó queda completa en la tabla.

## 5. Una ingesta completa en cinco pasos

Las tres funciones del lab siguen el mismo plan. Reciben el mes (las zonas no) y `reprocesar`, y hacen cinco pasos en este orden:

1. **Consultar** la tabla de control. Si ya hay un registro y `reprocesar` es falso, imprimir por qué no se hace nada y terminar.
2. **Descargar** el archivo desde la fuente original y guardarlo intacto en `<bronze>/_crudo/<tabla>/mes=AAAA-MM/lote=<id>/<archivo>`.
3. **Leer** el crudo con Spark y agregar los metadatos.
4. **Escribir** el lote en Parquet en `<bronze>/<tabla>/mes=AAAA-MM/lote=<id>/` con `io.escribir_lote`, que publica el marcador `_LOTE_COMPLETO` al final.
5. **Registrar** la ingesta en la tabla de control, con las filas que quedaron escritas.

Además de `mes` y `reprocesar`, cada función tiene tres parámetros con un valor por defecto. `base_bronze` es la carpeta raíz del bronze: hoy tu carpeta local, y `s3://<tu bucket>/bronze` cuando la use el DAG. `tabla` es el nombre de la carpeta en bronze y de la tabla en el control; las pruebas del lab usan otro nombre (que empieza con `prueba_`) para que sus registros no se mezclen con los tuyos. `obtener` es la función que descarga una URL a una ruta; por defecto es `bronze.descargar`, y las pruebas la reemplazan por una que copia un archivo pequeño para no usar la red.

Las pruebas de los tres ejercicios comprueban lo mismo: las rutas del crudo y del lote, las columnas, el contenido de los metadatos, que la huella coincida con la definición, que el registro quede en la tabla de control, que una segunda llamada con `reprocesar=False` no descargue ni escriba nada y que una tercera con `reprocesar=True` cree un lote nuevo y un registro nuevo. Para no ensuciar tu tabla de control, usan la tabla `prueba_*` de cada fuente y un mes de prueba, `2099-01`, que todavía no existe, y borran sus registros al terminar.

La celda siguiente prepara los archivos pequeños de prueba (un clima de 31 días, 12 zonas y 100 viajes, tomados de `tests/data`) y define la prueba común. No tienes que leerla para resolver los ejercicios.

In [ ]:
# Celda 15
from io import StringIO

MES_PRUEBA = "2099-01"                                  # un mes que todavía no existe
CARPETA_FIXTURES = Path(tempfile.mkdtemp(prefix="fixtures-c5-"))
FIXTURE_CLIMA = RAIZ / "tests" / "data" / "open_meteo_sample.json"
FIXTURE_ZONAS = RAIZ / "tests" / "data" / "taxi_zone_lookup_sample.csv"


def crear_parquet_de_viajes():
    """Convierte el CSV de 100 viajes de tests/data en un Parquet pequeño que hace de archivo de la fuente."""
    viajes_csv = (spark.read.option("header", True).option("inferSchema", True)
                  .csv(str(RAIZ / "tests" / "data" / "fhvhv_tripdata_sample.csv")))
    carpeta = CARPETA_FIXTURES / "viajes_parquet"
    viajes_csv.coalesce(1).write.mode("overwrite").parquet(str(carpeta))
    return sorted(carpeta.glob("part-*.parquet"))[0]


FIXTURE_VIAJES = crear_parquet_de_viajes()


def textos_y_huella(df, originales, cuantas=20):
    """Las primeras filas: sus columnas originales como texto y la huella que trae cada una."""
    columnas_texto = []
    for nombre in originales:
        columnas_texto.append(F.col(nombre).cast("string").alias(nombre))
    filas = []
    for fila in df.select(*columnas_texto, "_hash_fila").limit(cuantas).collect():
        valores = []
        for nombre in originales:
            valores.append(fila[nombre])
        filas.append((valores, fila["_hash_fila"]))
    return filas


def lotes_de(base, tabla, mes):
    """Nombres de las carpetas lote=... que hay en bronze para la tabla (y el mes)."""
    lotes = []
    for nombre in pio.listar_subcarpetas(pio.dir_fuente(base, tabla, mes)):
        if nombre.startswith(pio.PREFIJO_LOTE):
            lotes.append(nombre)
    return lotes


def llamar_con_aviso_de_prueba(llamar, f, reprocesar, base, tabla, obtener, limite, url):
    """Llama a tu función. En la prueba no hay descarga: se copia un archivo pequeño, y el mensaje lo dice."""
    salida = StringIO()
    try:
        with contextlib.redirect_stdout(salida):
            resultado = llamar(f, reprocesar, base, tabla, obtener, limite)
    finally:
        texto = salida.getvalue().replace(f"Descargando {url}", f"Usando archivo de prueba en lugar de {url}")
        print(texto, end="")
    return resultado


def probar_ingesta(f, caso, llamar):
    """Prueba común de las tres ingestas. `caso` describe la fuente; `llamar` invoca tu función con sus argumentos."""
    assert RESULTADO_LAB.get("E1 calcular_hash_fila") == "OK", "Primero resuelve el ejercicio 1 hasta que diga OK: la ingesta usa tu calcular_hash_fila."
    tabla, mes, nombre, url = caso["tabla"], caso["mes"], caso["nombre"], caso["url"]
    if mes is None:
        mes_control = MES_SIN_MES
    else:
        mes_control = mes
    originales = caso["originales"]

    descargas = []

    def obtener_de_prueba(url_pedida, destino):
        descargas.append((url_pedida, destino))
        pio.publicar(caso["archivo"], destino)

    def obtener_prohibido(url_pedida, destino):
        raise AssertionError("La segunda llamada (reprocesar=False) volvió a descargar: si la tabla de control ya tiene el registro, la función no debe descargar nada.")

    borrar_registros_de_prueba(tabla)                 # se parte sin registros de prueba
    try:
        with tempfile.TemporaryDirectory(prefix="bronze-prueba-") as base:
            # --- Primera llamada, reprocesar=False: ingesta normal.
            lote_1 = llamar_con_aviso_de_prueba(llamar, f, False, base, tabla, obtener_de_prueba, caso["limite"], url)
            assert lote_1 is not None, "La primera llamada debe devolver la ruta de la carpeta del lote que escribió (la función devuelve None solo cuando no hace nada)."
            lote_1 = str(lote_1)
            id_lote_1 = lote_1.rstrip("/").split("/")[-1][len(pio.PREFIJO_LOTE):]
            assert FORMATO_ID_LOTE.match(id_lote_1), f"El id del lote debe tener la forma AAAAMMDDTHHMMSSZ_xxxxxx y es {id_lote_1!r}."
            esperado = pio.dir_lote(base, tabla, id_lote_1, mes)
            assert lote_1 == esperado, f"El lote debe quedar en {esperado} y devolviste {lote_1}."
            assert len(descargas) == 1, f"La primera llamada debe descargar una vez y descargó {len(descargas)}."
            assert descargas[0][0] == url, f"Debe descargar desde la fuente original, {url}, y pidió {descargas[0][0]}."
            crudo_1 = ruta_crudo_lote(base, tabla, id_lote_1, nombre, mes)
            assert descargas[0][1] == crudo_1, f"El crudo debe guardarse en {crudo_1} y pediste guardarlo en {descargas[0][1]}."
            assert Path(crudo_1).exists(), f"No existe el archivo crudo {crudo_1}."
            assert (Path(lote_1) / pio.MARCA_LOTE_COMPLETO).exists(), f"El lote debe estar completo: falta el marcador {pio.MARCA_LOTE_COMPLETO} (lo publica pio.escribir_lote)."

            df_1 = spark.read.parquet(lote_1)
            assert df_1.columns == originales + COLUMNAS_METADATOS, f"Las columnas deben ser {originales + COLUMNAS_METADATOS} y tienes {df_1.columns}."
            filas_1 = df_1.count()
            assert filas_1 == caso["filas_con_limite"], f"El lote debe tener {caso['filas_con_limite']} filas y tiene {filas_1}."
            metadatos = df_1.select("_id_lote", "_fuente", "_archivo_origen").distinct().collect()
            assert len(metadatos) == 1, "Todas las filas del lote deben tener los mismos _id_lote, _fuente y _archivo_origen."
            assert metadatos[0]["_id_lote"] == id_lote_1, "_id_lote debe ser el mismo identificador de la carpeta del lote."
            assert metadatos[0]["_fuente"] == url, "_fuente debe ser la URL de donde se descargó."
            assert metadatos[0]["_archivo_origen"] == nombre, f"_archivo_origen debe ser el nombre del archivo crudo, {nombre}."
            assert dict(df_1.dtypes)["_fecha_ingesta"] == "timestamp", "_fecha_ingesta debe ser de tipo timestamp."
            huellas_1 = set()
            for valores, huella in textos_y_huella(df_1, originales):
                assert huella == huella_en_python(valores), "_hash_fila no coincide con la huella de las columnas originales (¿se calculó antes de agregar los metadatos?)."
            for fila in df_1.select("_hash_fila").collect():
                huellas_1.add(fila["_hash_fila"])

            registros = json.loads(consultar_watermark(tabla, "bronze", mes_control))
            assert len(registros) == 1, f"Después de la primera llamada la tabla de control debe tener 1 registro de ({tabla}, bronze, {mes_control}) y tiene {len(registros)}: revisa que registrar_watermark reciba la tabla y el mes de la función."
            registro = registros[0]
            assert registro["fuente"] == url, f"El registro debe llevar la fuente {url}."
            assert registro["capa"] == "bronze" and registro["tabla"] == tabla and registro["mes"] == mes_control, f"El registro debe ser de la capa bronze, la tabla {tabla} y el mes {mes_control}; tiene {registro['capa']}, {registro['tabla']} y {registro['mes']}."
            assert registro["filas"] == filas_1, f"El registro debe contar las filas que quedaron escritas ({filas_1}) y dice {registro['filas']}."
            assert registro["reprocesar"] is False, "El registro de la primera llamada debe decir reprocesar=False."

            # --- Segunda llamada, reprocesar=False: ya hay registro, no hace nada.
            salida = StringIO()
            with contextlib.redirect_stdout(salida):
                resultado_2 = llamar(f, False, base, tabla, obtener_prohibido, caso["limite"])
            assert resultado_2 is None, "Si ya hay registro y reprocesar es False, la función no hace nada y devuelve None."
            assert salida.getvalue().strip() != "", "Cuando no hace nada, la función debe imprimir por qué (ya hay un registro y reprocesar es False)."
            assert len(lotes_de(base, tabla, mes)) == 1, "La segunda llamada no debe escribir un lote nuevo."
            assert len(json.loads(consultar_watermark(tabla, "bronze", mes_control))) == 1, "La segunda llamada no debe registrar nada."

            # --- Tercera llamada, reprocesar=True: lote nuevo, crudo nuevo y registro nuevo.
            lote_3 = llamar_con_aviso_de_prueba(llamar, f, True, base, tabla, obtener_de_prueba, None, url)
            assert lote_3 is not None and str(lote_3) != lote_1, "Con reprocesar=True debe escribir un lote nuevo y devolver su carpeta."
            lote_3 = str(lote_3)
            id_lote_3 = lote_3.rstrip("/").split("/")[-1][len(pio.PREFIJO_LOTE):]
            assert len(descargas) == 2, f"Con reprocesar=True debe descargar otra vez (dos descargas en total) y van {len(descargas)}."
            crudo_3 = ruta_crudo_lote(base, tabla, id_lote_3, nombre, mes)
            assert descargas[1][1] == crudo_3 and Path(crudo_3).exists(), f"El crudo del lote nuevo debe quedar en {crudo_3}."
            assert Path(crudo_1).exists() and Path(lote_1).exists(), "Reprocesar no debe borrar el crudo ni el lote anteriores."
            assert len(lotes_de(base, tabla, mes)) == 2, "Después de reprocesar debe haber dos lotes."
            assert pio.lote_mas_reciente(base, tabla, mes) == id_lote_3, "El lote más reciente (pio.lote_mas_reciente) debe ser el que acabas de escribir."
            df_3 = spark.read.parquet(lote_3)
            assert df_3.count() == caso["filas_total"], f"Sin límite, el lote reprocesado debe tener las {caso['filas_total']} filas del archivo y tiene {df_3.count()}."
            huellas_3 = set()
            for fila in df_3.select("_hash_fila").collect():
                huellas_3.add(fila["_hash_fila"])
            assert huellas_1.issubset(huellas_3), "Las mismas filas ingestadas dos veces deben dar la misma huella: la huella no debe usar los metadatos."
            registros = json.loads(consultar_watermark(tabla, "bronze", mes_control))
            assert len(registros) == 2, f"Después de reprocesar la tabla de control debe tener 2 registros y tiene {len(registros)}."
            assert registros[-1]["reprocesar"] is True and registros[-1]["filas"] == caso["filas_total"], "El registro nuevo debe decir reprocesar=True y contar las filas del lote nuevo."
    finally:
        borrar_registros_de_prueba(tabla)             # no se deja ningún registro de prueba


def contar_dias_del_clima():
    return len(json.loads(FIXTURE_CLIMA.read_text(encoding="utf-8"))["daily"]["time"])


def contar_zonas():
    return len(FIXTURE_ZONAS.read_text(encoding="utf-8").strip().splitlines()) - 1


def contar_viajes():
    return spark.read.parquet(str(FIXTURE_VIAJES)).count()


LIMITE_DE_PRUEBA = 40
CASO_CLIMA = dict(
    tabla="prueba_clima", mes=MES_PRUEBA, nombre=pio.nombre_clima(MES_PRUEBA), url=bronze_pipeline.url_clima(MES_PRUEBA),
    archivo=FIXTURE_CLIMA, originales=["fecha", "lluvia_mm", "temperatura"],
    filas_con_limite=contar_dias_del_clima(), filas_total=contar_dias_del_clima(), limite=None)
CASO_ZONAS = dict(
    tabla="prueba_taxi_zones", mes=None, nombre=pio.NOMBRE_ZONAS, url=bronze_pipeline.URL_ZONAS,
    archivo=FIXTURE_ZONAS, originales=["LocationID", "Borough", "Zone", "service_zone"],
    filas_con_limite=contar_zonas(), filas_total=contar_zonas(), limite=None)
CASO_VIAJES = dict(
    tabla="prueba_fhvhv", mes=MES_PRUEBA, nombre=pio.nombre_viajes(MES_PRUEBA), url=url_viajes(MES_PRUEBA),
    archivo=FIXTURE_VIAJES, originales=spark.read.parquet(str(FIXTURE_VIAJES)).columns,
    filas_con_limite=LIMITE_DE_PRUEBA, filas_total=contar_viajes(), limite=LIMITE_DE_PRUEBA)
print("Archivos de prueba listos en", CARPETA_FIXTURES)
print(f"Clima: {CASO_CLIMA['filas_total']} días | zonas: {CASO_ZONAS['filas_total']} filas | viajes: {CASO_VIAJES['filas_total']} filas")

### Ejercicio 2 · Ingestar el clima: la función completa

Escribes tu código en la Celda 16 y la Celda 17 lo verifica.

**Qué hacer.** Completa `ingestar_clima(mes, reprocesar)`. La función ya trae el esqueleto de los cinco pasos y todo lo que no cambia de una fuente a otra: arma las rutas, calcula el identificador del lote, lee el JSON con Spark y lo convierte en una fila por día. Tú escribes cinco huecos, uno por paso:

1. Consultar la tabla de control con `consultar_watermark(tabla, "bronze", mes)`. El resto del paso ya está escrito: convierte el JSON y, si hay registros y `reprocesar` es falso, imprime por qué no hace nada y devuelve `None`.
2. Descargar el archivo con `obtener(url, crudo)`, que lo guarda intacto en la ruta del crudo del lote.
3. Agregar los metadatos al DataFrame de días con `agregar_metadatos`.
4. Escribir el lote con `pio.escribir_lote`, que guarda el Parquet y, solo si terminó bien, publica el marcador `_LOTE_COMPLETO`.
5. Registrar la ingesta con `registrar_watermark`, con las filas que quedaron escritas.

**Por qué importa.** Esta función junta todo el diseño del lab: no descarga lo que ya se trajo, guarda el crudo intacto, escribe un lote nuevo sin pisar los anteriores y deja constancia en la tabla de control solo cuando todo salió bien. Es la que el DAG llamará cada vez que llegue un mes nuevo.

La primera ejecución de la prueba tarda un poco: abre varias conexiones a Snowflake y usa la tabla `prueba_clima` y el mes `2099-01`.

<details>
<summary>Pista 1: el orden de los huecos</summary>

Los cinco huecos van en el orden en que ocurren las cosas: consultar la tabla, descargar, agregar los metadatos, escribir el lote y registrar. Si te pierdes, lee los comentarios `# Paso N` de la función: cada hueco está debajo del comentario de su paso.

</details>

<details>
<summary>Pista 2: pasos 1 y 2, consultar y descargar</summary>

`consultar_watermark` recibe tres textos, en este orden: la tabla, la capa y el mes, y devuelve un texto JSON que la función ya convierte en lista con `json.loads`. Para descargar, `obtener` recibe la URL de la fuente y la ruta donde guardar el archivo; las dos ya están calculadas arriba, en las variables `url` y `crudo`.

</details>

<details>
<summary>Pista 3: pasos 3, 4 y 5, metadatos, lote y registro</summary>

`agregar_metadatos(df, fuente, archivo_origen, id_lote, fecha_ingesta)` recibe el DataFrame de días, la URL, el nombre del archivo crudo, el identificador del lote y el momento de inicio, y todas esas variables ya existen. `pio.escribir_lote(df, carpeta_lote, archivos=1)` recibe el DataFrame con metadatos y la carpeta del lote. `registrar_watermark(fuente, capa, tabla, mes, filas, inicio, reprocesar)` recibe siete valores en ese orden; la fuente es la URL y la capa es el texto `"bronze"`.

</details>

In [ ]:
# Celda 16
def ingestar_clima(mes, reprocesar=False, base_bronze=BRONZE_ALUMNO, tabla=pio.CLIMA, obtener=bronze_pipeline.descargar):
    """Ingesta el clima de `mes` a bronze. Devuelve la carpeta del lote, o None si no hizo nada.

    base_bronze: carpeta raíz del bronze (local hoy; s3://<bucket>/bronze cuando la llame el DAG).
    tabla: nombre de la carpeta en bronze y de la tabla en la tabla de control (las pruebas usan otro).
    obtener: función obtener(url, destino) que descarga; las pruebas la reemplazan para no usar la red.
    """
    # Paso 1. Preguntarle a la tabla de control si este mes ya se ingestó.
    registros_json = ...  # TU CÓDIGO AQUÍ: consulta la tabla de control con consultar_watermark(tabla, "bronze", mes)
    registros = json.loads(registros_json)
    if len(registros) > 0 and not reprocesar:
        ultimo = registros[-1]
        print(f"No se ingesta {tabla} {mes}: la tabla de control ya tiene {len(registros)} registro(s), "
              f"el último del {ultimo['inicio']} UTC con {ultimo['filas']} filas. Usa reprocesar=True para ingestarlo otra vez.")
        return None

    # Un lote nuevo en cada ejecución: su momento de inicio (UTC), su identificador y sus rutas.
    inicio = datetime.now(timezone.utc).replace(microsecond=0)
    id_lote = bronze_pipeline.nuevo_id_lote(inicio)
    url = bronze_pipeline.url_clima(mes)
    nombre = pio.nombre_clima(mes)
    crudo = ruta_crudo_lote(base_bronze, tabla, id_lote, nombre, mes)
    carpeta_lote = pio.dir_lote(base_bronze, tabla, id_lote, mes)

    # Paso 2. Descargar desde la fuente original y guardar el crudo intacto en la carpeta de este lote.
    print(f"Descargando {url}")
    ...  # TU CÓDIGO AQUÍ: descarga la URL con obtener(url, crudo)

    # Paso 3. Spark lee el crudo (una fila por día) y se agregan los metadatos.
    json_crudo = spark.read.option("multiLine", True).json(crudo)
    dias = bronze_pipeline.clima_a_filas(json_crudo)
    con_metadatos = ...  # TU CÓDIGO AQUÍ: agrega los metadatos con agregar_metadatos(dias, url, nombre, id_lote, inicio)

    # Paso 4. Escribir el lote: Parquet y, al final, el marcador _LOTE_COMPLETO.
    ...  # TU CÓDIGO AQUÍ: escribe el lote con pio.escribir_lote(con_metadatos, carpeta_lote, archivos=1)

    # Paso 5. Registrar solo ahora, cuando todo salió bien, con las filas que quedaron escritas.
    filas = spark.read.parquet(carpeta_lote).count()
    ...  # TU CÓDIGO AQUÍ: registra con registrar_watermark(url, "bronze", tabla, mes, filas, inicio, reprocesar)
    print(f"Listo: {filas} filas en {carpeta_lote}")
    return carpeta_lote

In [ ]:
# Celda 17
def llamar_clima(f, reprocesar, base, tabla, obtener, limite):
    return f(MES_PRUEBA, reprocesar=reprocesar, base_bronze=base, tabla=tabla, obtener=obtener)


def prueba_e2(f):
    probar_ingesta(f, CASO_CLIMA, llamar_clima)


verificar("E2 ingestar_clima", ingestar_clima, prueba_e2)

### Ejercicio 3 · Ingestar las zonas: una fuente sin mes

Escribes tu código en la Celda 18 y la Celda 19 lo verifica.

**Qué hacer.** Completa `ingestar_zonas(reprocesar)`. Es el mismo plan de cinco pasos, con tres diferencias que importan. Las zonas son un solo archivo, sin mes: no hay carpeta `mes=...` ni en el crudo ni en el lote, y en la tabla de control el mes es el valor fijo `MES_SIN_MES` (el texto `sin_mes`). El archivo es un CSV con encabezado, que Spark lee con `option("header", True)` y deja todas las columnas como texto, igual que viene. Y el lote es pequeño (265 filas), así que se escribe en un solo archivo con `archivos=1`.

Esta vez tienes seis huecos: los cinco pasos de siempre y, en el paso 3, la lectura del CSV. Si lo escribiste bien en el ejercicio 2, aquí solo cambia el detalle de la fuente.

**Por qué importa.** Las zonas casi nunca cambian, pero se ingestan con el mismo rigor que los viajes: tienen su crudo por lote, sus metadatos y su registro. La tabla de control las distingue con `sin_mes`, y así la misma lógica de consulta sirve para fuentes con mes y sin mes.

<details>
<summary>Pista 1: qué cambia respecto al clima</summary>

Cambian tres cosas: el mes de la tabla de control es `MES_SIN_MES` en lugar de un mes; las rutas del crudo y del lote se arman sin mes (los dos últimos parámetros de `ruta_crudo_lote` y de `pio.dir_lote` se omiten); y el archivo se lee como CSV con encabezado.

</details>

<details>
<summary>Pista 2: leer el CSV</summary>

Spark lee un CSV con `spark.read.option("header", True).csv(ruta)`. Sin `inferSchema`, todas las columnas quedan como texto, que es lo que queremos en bronze: el dato tal como llegó. La ruta que debes leer es la del crudo del lote, la variable `crudo`.

</details>

<details>
<summary>Pista 3: los otros cuatro pasos</summary>

Son los mismos del ejercicio 2. La consulta y el registro usan `MES_SIN_MES` como mes. Los metadatos se agregan con `agregar_metadatos(original, url, nombre, id_lote, inicio)` y el lote se escribe con `pio.escribir_lote(con_metadatos, carpeta_lote, archivos=1)`.

</details>

In [ ]:
# Celda 18
def ingestar_zonas(reprocesar=False, base_bronze=BRONZE_ALUMNO, tabla=pio.ZONAS, obtener=bronze_pipeline.descargar):
    """Ingesta las zonas de taxi (un archivo, sin mes) a bronze. Devuelve la carpeta del lote, o None si no hizo nada.

    Los parámetros base_bronze, tabla y obtener son los mismos de ingestar_clima.
    """
    # Paso 1. Las zonas no tienen mes: en la tabla de control se registran con MES_SIN_MES.
    registros_json = ...  # TU CÓDIGO AQUÍ: consulta la tabla de control con consultar_watermark(tabla, "bronze", MES_SIN_MES)
    registros = json.loads(registros_json)
    if len(registros) > 0 and not reprocesar:
        ultimo = registros[-1]
        print(f"No se ingesta {tabla}: la tabla de control ya tiene {len(registros)} registro(s), "
              f"el último del {ultimo['inicio']} UTC con {ultimo['filas']} filas. Usa reprocesar=True para ingestarlo otra vez.")
        return None

    # Un lote nuevo en cada ejecución. Sin mes, las rutas no llevan la carpeta mes=...
    inicio = datetime.now(timezone.utc).replace(microsecond=0)
    id_lote = bronze_pipeline.nuevo_id_lote(inicio)
    url = bronze_pipeline.URL_ZONAS
    nombre = pio.NOMBRE_ZONAS
    crudo = ruta_crudo_lote(base_bronze, tabla, id_lote, nombre)
    carpeta_lote = pio.dir_lote(base_bronze, tabla, id_lote)

    # Paso 2. Descargar desde la fuente original y guardar el crudo intacto en la carpeta de este lote.
    print(f"Descargando {url}")
    ...  # TU CÓDIGO AQUÍ: descarga la URL con obtener(url, crudo)

    # Paso 3. Spark lee el CSV con encabezado (todas las columnas como texto) y se agregan los metadatos.
    original = ...  # TU CÓDIGO AQUÍ: lee el CSV crudo con encabezado (option("header", True))
    con_metadatos = ...  # TU CÓDIGO AQUÍ: agrega los metadatos con agregar_metadatos(original, url, nombre, id_lote, inicio)

    # Paso 4. Escribir el lote en un solo archivo.
    ...  # TU CÓDIGO AQUÍ: escribe el lote con pio.escribir_lote(con_metadatos, carpeta_lote, archivos=1)

    # Paso 5. Registrar solo al terminar bien.
    filas = spark.read.parquet(carpeta_lote).count()
    ...  # TU CÓDIGO AQUÍ: registra con registrar_watermark(url, "bronze", tabla, MES_SIN_MES, filas, inicio, reprocesar)
    print(f"Listo: {filas} filas en {carpeta_lote}")
    return carpeta_lote

In [ ]:
# Celda 19
def llamar_zonas(f, reprocesar, base, tabla, obtener, limite):
    return f(reprocesar=reprocesar, base_bronze=base, tabla=tabla, obtener=obtener)


def prueba_e3(f):
    probar_ingesta(f, CASO_ZONAS, llamar_zonas)


verificar("E3 ingestar_zonas", ingestar_zonas, prueba_e3)

### Ejercicio 4 · Ingestar los viajes, con un modo de prueba

Escribes tu código en la Celda 20 y la Celda 21 lo verifica.

**Qué hacer.** Completa `ingestar_viajes(mes, reprocesar)`. Es el mismo plan de cinco pasos con un archivo Parquet por mes. Lo nuevo es que el archivo pesa unos 450 MiB, y escribir bronze con millones de filas en una máquina de dos núcleos tarda demasiado para una clase. Por eso la función tiene un parámetro `limite_filas`. Con su valor normal, `None`, escribe todas las filas. Con un número, por ejemplo `1000`, **descarga el archivo completo** y guarda el crudo intacto, pero escribe en bronze solo las primeras `limite_filas` filas. Así pruebas todo el recorrido (la descarga real, la huella, el lote, el marcador y el registro) sin esperar. El archivo crudo queda completo y la tabla de control cuenta las filas que de verdad se escribieron.

Un lote limitado no es un mes ingestado, y marcarlo así en la tabla de control sería una mentira: el DAG vería «ya hay registro» y se saltaría un mes que en realidad tiene mil filas. Por eso la función ya trae una guarda que solo permite `limite_filas` con una tabla de prueba (un nombre que empiece con `prueba`), y la tuya no la tienes que escribir.

Tienes seis huecos: los cinco pasos y, en el paso 3, la lectura del Parquet crudo. El límite ya está escrito debajo de ese hueco.

**Por qué importa.** Los viajes son el volumen real del pipeline. Que la misma función sirva para una prueba de mil filas en tu máquina y para el mes completo en Glue es lo que permite probar el recorrido sin esperar horas, y la guarda evita que una prueba contamine el control de la ingesta real.

<details>
<summary>Pista 1: qué cambia respecto al clima</summary>

Casi nada. Los viajes tienen mes, así que las rutas y la tabla de control se arman igual que en el clima. La URL sale de `url_viajes(mes)`, el nombre del archivo de `pio.nombre_viajes(mes)`, y el archivo crudo es un Parquet que Spark lee directamente.

</details>

<details>
<summary>Pista 2: leer el Parquet</summary>

`spark.read.parquet(ruta)` lee un Parquet, y la ruta es la del crudo del lote, la variable `crudo`. Debajo del hueco, el código ya recorta las filas cuando `limite_filas` tiene un valor.

</details>

<details>
<summary>Pista 3: escribir el lote</summary>

Los viajes tienen millones de filas, así que no se juntan en un solo archivo: `pio.escribir_lote(con_metadatos, carpeta_lote)` sin el parámetro `archivos` deja que Spark reparta la escritura entre sus particiones.

</details>

In [ ]:
# Celda 20
def ingestar_viajes(mes, reprocesar=False, base_bronze=BRONZE_ALUMNO, tabla=pio.VIAJES, obtener=bronze_pipeline.descargar,
                    limite_filas=None):
    """Ingesta los viajes de `mes` a bronze. Devuelve la carpeta del lote, o None si no hizo nada.

    Los parámetros base_bronze, tabla y obtener son los mismos de ingestar_clima.
    limite_filas: modo de prueba. Descarga el archivo completo y guarda el crudo intacto, pero escribe en bronze
    solo las primeras `limite_filas` filas. Solo se permite con una tabla de prueba (nombre que empieza con «prueba»).
    """
    # Guarda: un lote recortado nunca debe quedar registrado como si fuera el mes completo.
    if limite_filas is not None and not tabla.startswith(watermark.PREFIJOS_DE_PRUEBA):
        raise ValueError(f"limite_filas solo se puede usar con una tabla de prueba (que empiece con 'prueba'); se pidió la tabla {tabla!r}.")

    # Paso 1. Preguntarle a la tabla de control si este mes ya se ingestó.
    registros_json = ...  # TU CÓDIGO AQUÍ: consulta la tabla de control con consultar_watermark(tabla, "bronze", mes)
    registros = json.loads(registros_json)
    if len(registros) > 0 and not reprocesar:
        ultimo = registros[-1]
        print(f"No se ingesta {tabla} {mes}: la tabla de control ya tiene {len(registros)} registro(s), "
              f"el último del {ultimo['inicio']} UTC con {ultimo['filas']} filas. Usa reprocesar=True para ingestarlo otra vez.")
        return None

    # Un lote nuevo en cada ejecución: su momento de inicio (UTC), su identificador y sus rutas.
    inicio = datetime.now(timezone.utc).replace(microsecond=0)
    id_lote = bronze_pipeline.nuevo_id_lote(inicio)
    url = url_viajes(mes)
    nombre = pio.nombre_viajes(mes)
    crudo = ruta_crudo_lote(base_bronze, tabla, id_lote, nombre, mes)
    carpeta_lote = pio.dir_lote(base_bronze, tabla, id_lote, mes)

    # Paso 2. Descargar el archivo completo desde la fuente original y guardar el crudo intacto.
    print(f"Descargando {url}")
    ...  # TU CÓDIGO AQUÍ: descarga la URL con obtener(url, crudo)

    # Paso 3. Spark lee el Parquet crudo; en modo de prueba se recorta a las primeras filas; se agregan los metadatos.
    original = ...  # TU CÓDIGO AQUÍ: lee el Parquet crudo con spark.read.parquet(crudo)
    if limite_filas is not None:
        original = original.limit(limite_filas)
    con_metadatos = ...  # TU CÓDIGO AQUÍ: agrega los metadatos con agregar_metadatos(original, url, nombre, id_lote, inicio)

    # Paso 4. Escribir el lote; con muchas filas, Spark lo reparte en varios archivos.
    ...  # TU CÓDIGO AQUÍ: escribe el lote con pio.escribir_lote(con_metadatos, carpeta_lote)

    # Paso 5. Registrar solo al terminar bien, con las filas que de verdad quedaron escritas.
    filas = spark.read.parquet(carpeta_lote).count()
    ...  # TU CÓDIGO AQUÍ: registra con registrar_watermark(url, "bronze", tabla, mes, filas, inicio, reprocesar)
    print(f"Listo: {filas} filas en {carpeta_lote}")
    return carpeta_lote

In [ ]:
# Celda 21
def llamar_viajes(f, reprocesar, base, tabla, obtener, limite):
    return f(MES_PRUEBA, reprocesar=reprocesar, base_bronze=base, tabla=tabla, obtener=obtener, limite_filas=limite)


def prueba_e4(f):
    # La guarda del modo de prueba: con la tabla real, limite_filas debe fallar antes de tocar nada.
    def obtener_prohibido(url_pedida, destino):
        raise AssertionError("La guarda debe rechazar limite_filas con la tabla real antes de descargar.")

    with tempfile.TemporaryDirectory(prefix="bronze-prueba-") as base:
        try:
            f(MES_PRUEBA, base_bronze=base, tabla=pio.VIAJES, obtener=obtener_prohibido, limite_filas=10)
        except ValueError:
            pass
        else:
            raise AssertionError("Con la tabla real (fhvhv), limite_filas debe lanzar un ValueError: no borres la guarda del inicio de la función.")
    probar_ingesta(f, CASO_VIAJES, llamar_viajes)


verificar("E4 ingestar_viajes", ingestar_viajes, prueba_e4)

## 6. Ejecuta tus ingestas con datos reales

Hasta aquí las funciones se probaron con archivos pequeños. La celda siguiente las ejecuta de verdad, con descargas reales, sobre tu carpeta `~/datos_clase05/bronze_alumno`. Usa las tablas de práctica `prueba_clima`, `prueba_taxi_zones` y `prueba_fhvhv`, para que lo que hagas hoy no quede registrado como si fuera la ingesta real: esa la ejecutará el DAG de la clase 6 con las tablas de verdad. Para los viajes usa el modo de prueba, así que **descarga el archivo completo de enero (unos 450 MiB)** y escribe en bronze las primeras 1 000 filas.

Con el clima ves los tres casos seguidos: la primera llamada ingesta, la segunda no hace nada porque la tabla de control ya tiene el mes, y la tercera, con `reprocesar=True`, descarga otra vez y crea un lote nuevo. Los viajes tardan lo que tarde la descarga.

In [ ]:
# Celda 22
ejercicios_de_la_ingesta = ["E1 calcular_hash_fila", "E2 ingestar_clima", "E3 ingestar_zonas", "E4 ingestar_viajes"]
pendientes = []
for nombre in ejercicios_de_la_ingesta:
    if RESULTADO_LAB.get(nombre) != "OK":
        pendientes.append(nombre)

TABLAS_DE_PRACTICA = ["prueba_clima", "prueba_taxi_zones", "prueba_fhvhv"]

if pendientes:
    print("Todavía no: resuelve y verifica los ejercicios", pendientes, "(deben decir OK) y vuelve a ejecutar esta celda.")
else:
    for tabla_practica in TABLAS_DE_PRACTICA:
        borrar_registros_de_prueba(tabla_practica)            # se parte de cero

    print("=== Clima de enero, primera llamada (reprocesar=False): ingesta")
    ingestar_clima("2024-01", tabla="prueba_clima")
    print("\n=== Clima de enero, segunda llamada (reprocesar=False): no hace nada")
    ingestar_clima("2024-01", tabla="prueba_clima")
    print("\n=== Clima de enero, tercera llamada (reprocesar=True): descarga otra vez y crea otro lote")
    ingestar_clima("2024-01", reprocesar=True, tabla="prueba_clima")

    print("\n=== Zonas, primera llamada y segunda llamada")
    ingestar_zonas(tabla="prueba_taxi_zones")
    ingestar_zonas(tabla="prueba_taxi_zones")

    print("\n=== Viajes de enero en modo de prueba: descarga el archivo completo y escribe 1 000 filas")
    with cronometro("viajes de enero, modo de prueba"):
        ingestar_viajes("2024-01", tabla="prueba_fhvhv", limite_filas=1000)
    print("\n=== Viajes de enero, segunda llamada: no hace nada")
    ingestar_viajes("2024-01", tabla="prueba_fhvhv", limite_filas=1000)

**Qué observar.** Las llamadas que no hacen nada imprimen su motivo y no descargan ni escriben. La tercera llamada del clima crea un segundo lote del mismo mes. Dentro de la celda, la segunda llamada de cada fuente se salta porque la tabla de control ya tiene el registro: eso es justamente lo que queremos que haga el DAG cuando lo ejecuten dos veces. Ejecuta esta celda **una sola vez**: empieza borrando los registros de práctica, así que si la repites vuelve a empezar de cero y descarga otra vez los 450 MiB de viajes.

La celda siguiente lista lo que quedó en tu bronze, muestra los registros de la tabla de control de cada fuente y lee un lote de cada una.

In [ ]:
# Celda 23
if RESULTADO_LAB.get("E4 ingestar_viajes") != "OK" or "TABLAS_DE_PRACTICA" not in globals():
    print("Primero ejecuta la celda anterior con los cuatro ejercicios en OK.")
else:
    raiz = Path(BRONZE_ALUMNO)
    print("Archivos en tu bronze:")
    for archivo in sorted(raiz.rglob("*")):
        es_oculto_de_spark = archivo.name.startswith(".") or archivo.name.endswith(".crc")
        if archivo.is_file() and not es_oculto_de_spark:
            print(f"  {archivo.relative_to(raiz)}  ({archivo.stat().st_size:,} bytes)")

    print("\nTabla de control:")
    consultas = [("prueba_clima", "2024-01"), ("prueba_taxi_zones", MES_SIN_MES), ("prueba_fhvhv", "2024-01")]
    for tabla_practica, mes_control in consultas:
        registros = json.loads(consultar_watermark(tabla_practica, "bronze", mes_control))
        print(f"  {tabla_practica} / {mes_control}: {len(registros)} registro(s)")
        for registro in registros:
            print(f"    inicio={registro['inicio']}  filas={registro['filas']}  reprocesar={registro['reprocesar']}")

    print("\nClima: el lote más reciente, con sus metadatos")
    lote_clima = pio.dir_lote_reciente(BRONZE_ALUMNO, "prueba_clima", "2024-01")
    spark.read.parquet(lote_clima).select("fecha", "lluvia_mm", "temperatura", "_hash_fila", "_id_lote").show(3, truncate=24)
    print("Viajes: tres columnas del lote, con la huella")
    lote_viajes = pio.dir_lote_reciente(BRONZE_ALUMNO, "prueba_fhvhv", "2024-01")
    spark.read.parquet(lote_viajes).select("pickup_datetime", "PULocationID", "_hash_fila", "_id_lote").show(3, truncate=24)

**Qué observar.** En `_crudo/` hay un archivo por lote: dos del clima, uno de las zonas y el de viajes de unos 450 MiB, cada uno con su identificador. En el clima hay dos carpetas `lote=...` dentro de `mes=2024-01`, con la misma información y distinto `_id_lote`; leídas juntas darían cada día repetido, y la huella es lo que permitirá quitar esa repetición en la clase 6. La tabla de control cuenta dos registros del clima (el segundo con `reprocesar=True`) y uno de cada una de las otras fuentes. El lote de viajes tiene exactamente 1 000 filas, aunque el crudo trae millones.

### Un mes completo, si quieres

La ingesta real de un mes completo de viajes es opcional. Escribir millones de filas con Spark en dos núcleos tarda bastante más que la prueba y llena el disco con otro archivo de 450 MiB, y por eso viene apagada. Si quieres verla, cambia `EJECUTAR_MES_COMPLETO` a `True` en la celda siguiente. Usa la tabla `prueba_fhvhv` y febrero, para que tampoco marque ese mes como ingestado en la tabla real.

In [ ]:
# Celda 24
EJECUTAR_MES_COMPLETO = False

if EJECUTAR_MES_COMPLETO and RESULTADO_LAB.get("E4 ingestar_viajes") == "OK":
    with cronometro("viajes de febrero, mes completo"):
        ingestar_viajes("2024-02", tabla="prueba_fhvhv")
else:
    print("Mes completo apagado: cambia EJECUTAR_MES_COMPLETO a True para ejecutarlo.")

## 7. Cierre: lo que llevas a la clase 6

Estas tres funciones son las que el DAG de la clase 6 llamará con el mes y `reprocesar`. El DAG no sabrá nada de URLs, de Spark ni de lotes: solo dirá «ingesta el mes 2024-01» o «reprocesa el mes 2024-01», y cada función decidirá, consultando la tabla de control, si descarga o se salta el mes. Esa es la propiedad que se busca en un pipeline que corre cada mes: ejecutarlo dos veces seguidas es seguro, y reprocesar es una decisión explícita que no borra la historia.

Dos cosas quedan pendientes y las atiende la clase 6. La primera es que en el DAG `base_bronze` valdrá `s3://<tu bucket>/bronze` en lugar de tu carpeta local. La segunda es que dentro de Glue hay que resolver cómo llega la conexión a Snowflake para que las funciones de la tabla de control puedan registrar. Las ingestas que acabas de escribir, junto con los lotes que dejaste en `~/datos_clase05/bronze_alumno`, serán también el punto de partida del lab de silver y gold.

Al terminar, la celda siguiente borra los registros de práctica de la tabla de control (los archivos de tu bronze se quedan, porque el lab de la clase 6 los usa), borra los archivos de prueba temporales y apaga Spark. Después de ejecutarla, las celdas que usan `spark` dejan de funcionar hasta que reinicies el kernel y las vuelvas a correr desde la sección 0.

In [ ]:
# Celda 25
for tabla_practica in ["prueba_clima", "prueba_taxi_zones", "prueba_fhvhv", "demo"]:
    borradas = borrar_registros_de_prueba(tabla_practica)
    print(f"Registros de {tabla_practica} borrados de la tabla de control: {borradas}")

shutil.rmtree(CARPETA_FIXTURES, ignore_errors=True)
print("\nResultado de los ejercicios:")
for nombre, estado in RESULTADO_LAB.items():
    print(f"  {nombre}: {estado}")

spark.stop()
print("\nSpark cerrado. Ya puedes usar la terminal.")